# The construction of the program (Versions 3.X)

The program is written in Python. It generates SINGULAR scripts (reflecting the mutation and crossover of the genes) and evokes the SINGULAR computation with the generated scripts. Then the program retrieves the result of the SINGULAR computation and prepares the new genes through the genetic algorithm.

# How are the algorithm of Mutatation and Crossover?

## Mutation are applied to the following items in the genes.

1. Polynomial (Possible, Not tested, and currently not done)
2. Divisor (A weight vector over rational divisors)
3. Rational places for computing code vectors: The maximal size of admissible rational places is dependent on the divisor, for the rational places and the divisor must be disjoint. 
4. Adjustment of Divisors so that the condition 2g-2<deg(D)<"size of rational places" is satisfied.
5. Chosen rows of the classical code generation matrix for CSS construction. 

- If an item in the list  is chosen for mutation, and the subsequent items must also be altered. We call it repairment. The repairment is an essential part of the operation of the function Mutation.

- We also intriduce the locking on the specified items. If an item is locked, it cannot be modified by the Mutation. Otherwise, it is rewritable.

## Crossver of genes G1 and G2 are applined to the following items in the genes.

1. New polynomial := G1.Polynomial (after scission)  + G2.Polynomial (after scission). But not implemented yet.
1. New Divisor := A uniform crossing of G1.Divisor and G2.Divisor
1. New Rational Places: Randomly chosen a fixed number of elements from the union of the rational places of G1 and G2.   

## The Singular libraries are necessary, in addition to the main body of the Python program.

We use SINGULAR and its library (broneth.lib) to generate classical error correction codes. But the library does not suffice to generate quantum error correction codes, and we prepare additional functions for CSS constructions. Those functions are written by the SINGULAR programming style. 

## The visualization of the result is enable. 

We prepare another notebook. Using it, we can read the output cells of this notebook for analysing and visualizing the GA flow. 

## N.B. Mutation (with an different order) can also be applied to the genes. 

1. Polynomial (Possible, Not tested, and currently not done)
2. Rational places for computing code vectors: the maximal size of admissible rational places should be fixed first.
3. Divisors: they do not conflit the rational places and the condition 2g-2<deg(D)<"size of rational places" is satisfied.
4. Chosen rows of the classical code generation matrix for CSS construction. 

The mutation with the above order is implemented in another notebook. (Versions 7.X)



In [1]:
import random
import subprocess
import time
import copy
import sympy

def RANDOM_POLY(r,DMAX,length):
    CHECK_POLY="ring r=2,(x,y),lp;\n"
    x,y=sympy.symbols("x y")
    monomials=[]
    for i in range(0,DMAX):
        for j in range(0,DMAX):
            monomials.append(x**i*y**j)
    monomials=random.sample(monomials,length)
    
    f=0*x
    for u in monomials:
        f+=random.randint(1,r^2-1)*u

    CHECK_POLY+="poly f="+str(f)+";f;\n" 
    CHECK_POLY+="ideal I=f,jacob(f);\n"
    CHECK_POLY+="I=std(I);"
    CHECK_POLY+="print(\"dim(I)=\"+string(dim(I)));\n"
    CHECK_POLY+="list F=factorize(f);F;print(\"factor=\"+string(size(F[2])));\n"
    CHECK_POLY+="print(\"JUDGE=\"+string( dim(I)==0 & size(F[2])==2));\n"
    si=open("CHECK_POLY.txt","w")
    si.write(CHECK_POLY)
    si.close()
    result_HC = subprocess.run(["Singular -q CHECK_POLY.txt"], shell=True,capture_output=True, text=True)
    #print(int(result_HC.stdout[-2])==1)
    return f,int(result_HC.stdout[-2])==1



    

In [16]:
from dataclasses import dataclass
import random
import subprocess
import time
import copy
import sympy
import numpy as np

@dataclass
class Config:
    learning_rate: float = 0.01
    epochs: int = 10
    batch_size: int = 32
    size_D: int =7
    crossover_rate: float = 0.8
    mutation_rate: float = 0.9
    mutation_strength: int = 1
config = Config()


##
## A Gene is defined by a Singular input written in several separated texts. 
## A0: The basic Input computing up to the classical AG code matrix C.
## C:  The extended input, succeeding the comptation by A0 and consting a CSS code.
##  
## The key parts in A0 are the two intvecs G (the divisor defining the linear system), D (the places for computaing the code vectors), the formula of the curve, and the ring.
## The key part in C is the intvec m that determines several row vectors in C for constructing the CSS code. 
## These data are treated as variables and rewritten by the random sampling.
##  
## TODO: The gene data (A0+C) could be replaced by the contents of "AT_ALL_SUBMAT.txt", which examines all possibilities of the intvec m.
##

A0=\
"LIB \"mybrnoeth.lib\";\n\
LIB \"myprocs.txt\";\n\
LIB \"AinV.cpp\";\n\
ring s=2,(x,y),lp;\n\
list HC=Adj_div(x3+y2+y);\n\
HC=NSplaces(1..2,HC);\n\
HC=extcurve(2,HC);\n\
def ER=HC[1][4];\n\
setring ER;\n\
intvec G=5;\n\
intvec D=2,3,4,5,6,7,8,9; \n\
matrix C=AGcode_L(G,D,HC);\n\
print(C);\n\
print(\"Hamming_wt\");\n\
print(min_wt_rmat(C));\n\
"
C="print(\"C\");\
print(C);\
print(\"check_fully_one_rows\");\
intvec a=check_fully_one_rows(C);\
print(a);\
if (nrows(C)>1){C=MySubmat(a,C);\
print(\"C [The row (1,1,...,1) is removed]\");\
print(C);}\
print(\"Check_fully_zero_columns\");\
print(check_fully_zero_columns(C));\
intvec b=check_fully_zero_columns(C);\
//if (1){C=MySubmat_cols(b, C);\
//print(\"C [Fully zero colums are removed.]\");\
//print(C);//};\
print(my_min_wt_rmat(C));\
print(\"SC\");\
intvec m=1..nrows(C);\
m=shuffle(m,size(m));\
m=m[1],m[2];\
print(m);\
matrix SC=MySubmat(m,C);\
print(SC);\
print(\"Ker(C)\");\
print(MyKer(C));\
print(\"Ker(SC)\");\
print(MyKer(SC));\
matrix KC=MyKer(C);\
matrix KSC=MyKer(SC);\
print(\"KSC\\KC\");\
print(MySupplement(KC,KSC));\
print(my_min_wt_rmat(MySupplement(KC,KSC)));\
print(\"C\\SC\");\
print(MySupplement(SC,C));\
print(my_min_wt_rmat(MySupplement(SC,C)));\
matrix CONCAT=concat(transpose(MySupplement(KC,KSC)),transpose(MySupplement(SC,C)));\
CONCAT=transpose(CONCAT);\
print(\"KSC\\KC + C\\SC\");\
print(CONCAT);\
print(\"rank(CONCAT)=\"+string(mat_rank(CONCAT)));\
print(my_min_wt_rmat(CONCAT));\
quit;"
C=C.replace(";",";\n")

##
##  The python class defining a gene
##
##  When the text data of a gene is provided, this instance is initialized.
##  For the later usage, the data of Places [(degrees, index)...] , the total number of rational places, and the genus are estimated at the initialization.  
##
class Gene:
    def __init__(
        self,
        gene_text : str  
    ):
        self.gene_text=gene_text
        self.success=None
        self.distance=None
        self.NrRatpl=None
        self.PLACES=None
        self.Get_NrRatpl_PLACES()
        self.lock=dict()
        self.lock["ring s"]=False
        self.lock["Adj_div"]=False
        self.lock["intvec G"]=False
        self.lock["intvec D"]=False
        self.lock["intvec m"]=False
        self.important_gene_text=dict()
    def get_distance(self,i):
        if i==None:
            i=-9999
        #print("get_distance",i)
        self.distance=i
    def get_result(self,txt):
        self.result=txt
    def show_gene(self):
        print(self.gene_text)
    def show_polynomial(self):
        if self.gene_text!=None:
            polynomial=[u for u in self.gene_text.split("\n") if u.find("Adj_div")>=0][-1]
            return polynomial.split("(")[1].split(")")[0]
    def show_divisor(self):
        if self.gene_text!=None:
            divisor=[u for u in self.gene_text.split("\n") if u.find("intvec G")>=0][-1]
            return eval("["+divisor.split("=")[1].split(";")[0]+"]")
    def show_places(self):
        if self.gene_text!=None:
            places=[u for u in self.gene_text.split("\n") if u.find("intvec D")>=0][-1]
        return eval("["+places.split("=")[1].split(";")[0]+"]")
    def show_CSS_rows(self):
        if self.gene_text!=None:
            CSS_rows=[u for u in self.gene_text.split("\n") if u.find("intvec m")>=0][-1]
        return eval("["+CSS_rows.split("=")[1].split(";")[0]+"]")
    def Get_NrRatpl_PLACES(self):
        AS=self.gene_text.split("\n") 
        f=open("A_HC.txt","w")
        for u in AS:
            if u.find("def ER=")>=0:
                break
            print(u,file=f)
        print("print(\"---HC----\");HC[3];print(\"---HC----\");",file=f)
        f.close()
        result_HC = subprocess.run(["Singular -q A_HC.txt"], shell=True,capture_output=True, text=True)
        NrRatpl=None
        PLACES=[]
        HCFOUND=0
        for u in result_HC.stdout.split("\n"):
            #print(u)
            if u.find("NrRatPl")>=0:
                NrRatpl=u
            if u.find("---HC---")>=0:
                HCFOUND+=1
            if HCFOUND==1:
                PLACES.append(u)
        
        NrRatpl=int(NrRatpl.split()[-1])
        
        PLACES=[eval("("+u+")") for u in PLACES[1:] if u.find(":")<0]
        self.NrRatpl=NrRatpl
        #PLACES.sort()
        self.PLACES=PLACES
        return NrRatpl,PLACES
    def runCSS(self):
        self.success=False
        fo=open("AW.txt","w")
        fo.write(self.gene_text)
        fo.close()
        result = subprocess.run(["Singular -q AW.txt"], shell=True,capture_output=True, text=True)
        self.get_result(result.stdout)
        if result.stdout.find("Vector basis successfully computed")>=0:
            #print("OK")
            self.success=True
            self.get_result(result.stdout)
            self.get_distance(int(result.stdout.split("\n")[-2]))
            OK=True
            pass
            #OK=False
            #G.get_result(result.stdout)
            #G.get_distance=None
            #print("FAILED")
            #f=open("AT_.txt","r")
            #print(f.readline())
            #f.close()        
    def RandomCSS(self,randomized=True):
        NrRatpl,PLACES=self.Get_NrRatpl_PLACES()
        AS=self.gene_text.split("\n")
        #print(AS)
        if randomized==True:
            OK=False
            while OK==False:
                AT=open("AT_.txt","w")
                iavoid=None
                for u in AS:
                    if u.find("intvec G")>=0 and randomized==True:
                        randomg,iavoid=getrandomG(1,4)
                        #print(randomg,iavoid)
                        w=str(randomg).replace("]","").replace("[","")
                        print("intvec G="+w+";",file=AT)   
                        #print("intvec G="+w+";",iavoid)   
                    elif u.find("intvec D")>=0 and randomized==True:
                        w=str(randomD(1,9,6,iavoid)).replace("]","").replace("[","")
                        #print(w)
                        print("intvec D="+w+";",file=AT)
                        #print("intvec D="+w+";")
                    else:
                        print(u,file=AT)
                AT.close()
            
                
                # コマンド実行
                result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
                
                #print(result.stdout)   # 標準出力
                if result.stdout.find("Vector basis successfully computed")>=0:
                    #print("OK")
                    OK=True
                    self.success=True
                else:
                    pass
                    OK=False
                    #print("FAILED")
                    f=open("AT_.txt","r")
                    #print(f.readline())
                    f.close()
                if result.stdout.find("wrong range")>=0:
                    break
                #print(result.stderr)   # エラー出力
                #print(result.returncode)  # 終了コード
        else:
            AT=open("AT_.txt","w")
            iavoid=None
            for u in AS:
                print(u,file=AT)
            AT.close()
            result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
            
            #print(result.stdout)   # 標準出力
            if result.stdout.find("Vector basis successfully computed")>=0:
                #print("OK")
                self.success=True
                OK=True
            else:
                pass
                #OK=False
                #print("FAILED")
                #f=open("AT_.txt","r")
                #print(f.readline())
                #f.close()
        if result.stdout.find("Vector basis successfully computed")>=0:
            #print("OK")
            self.success=True
            G.get_result(result.stdout)
            G.get_distance(int(result.stdout.split("\n")[-2]))
            OK=True
        else:
            pass
            #OK=False
            #G.get_result(result.stdout)
            #G.get_distance=None
            #print("FAILED")
            #f=open("AT_.txt","r")
            #print(f.readline())
            #f.close()



def disj_divs (H,P,auxIV,NrRatpl,PLACES):
# USGE:
#    For example,
#    H=[1, 0, 0, 0, 1] : G, INDEX TO PLACES (AS A LIST) 
#    P=[1, 6] : D, INDEX TO POINTS.
#    PLACES=[(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)] : List [...(deg., seq num.)..] (
#    POINTS ARE THE REARANGEMENT OF PLACES WHERE A PLACE (of degree d) YIELDS d points. 
#    viz., POINTS=[1|2|3|4,5|6,7|8,9]
#    THEREFORE H REFERS TO POINT 1, 6, 7.
#   The function disj_divs CHECKS WHETHER H AND P HAVE OVERLAPS ON SOME POINTS.
#   ALSO IT RETURNS THE POINTS CORRESPONDING TO THE GIVEN H, THE PLACES CORRESPONDING TO THE GIVEN H, THE PLACES CORRESPONDING TO THE GIVEN P.
#   BY COMPARING THOSE DATA, WE CAN SHAVE OFF THE OVERLAP BETWEEN H AND P. 

    if auxIV!=None:
        auxIVD=dict()
        for u in PLACES:
            auxIVD[u[0]]=0
        for u in PLACES:
            auxIVD[u[0]]+=1
        auxIV=[auxIVD[u] for u in auxIVD.keys()]
    s1=len(H);
    s2=len(P);
    s=2 
    #print("H (G)="+str(H));
    #print("P (POINTS)="+str(P));
    #print("PLACES="+str(PLACES));
    #print("s="+str(s));
    PLACES_POINT=dict()
    for u in PLACES:
        PLACES_POINT[u]=list()
    DIVISION=[u[0] for u in PLACES]
    #print(DIVISION)
    counter=1
    for u,v in zip(PLACES,DIVISION):
        for _ in range(v):
            PLACES_POINT[u].append(counter)
            counter+=1
    #print("PLACES->POINT",PLACES_POINT)    
    PLACES_POINT_KEYS=list(PLACES_POINT.keys())
    PLACES_POINT_KEYS.sort()
    POINTS_PLACE=dict()
    for u in PLACES_POINT.keys():
        for w in PLACES_POINT[u]:
            POINTS_PLACE[w]=u
    #print("POINTS->PLACE",POINTS_PLACE)

    H_CORRESPONDING_POINTS=list()
    H_CORRESPONDING_PLACES=list()
    for u in range(len(H)):
        if H[u]>0:
            ky=PLACES[u]
            H_CORRESPONDING_POINTS.extend(PLACES_POINT[ky])
            H_CORRESPONDING_PLACES.append(ky)
    #print("H->POINTS:",H_CORRESPONDING_POINTS)
    #print("H->PLACES:",H_CORRESPONDING_PLACES)
    
    P_CORRESPONDING_PLACES=list()
    for u in P:
        P_CORRESPONDING_PLACES.append(POINTS_PLACE[u])
    P_CORRESPONDING_PLACES=list(set(P_CORRESPONDING_PLACES))
    P_CORRESPONDING_PLACES.sort()
    #print("P->PLACES:",P_CORRESPONDING_PLACES)      
    return len(list( set(P) & set(H_CORRESPONDING_POINTS)))>0,H_CORRESPONDING_POINTS,H_CORRESPONDING_PLACES,P_CORRESPONDING_PLACES
        


def PallowedByH(HasPO,NrRatpl):
#
# Returns the possible rational places (for constructing the linear system), provided that the PLACES (for computing code vectors) are given.
#
    POpossible=[u+1 for u in range(NrRatpl)]
    return list(set(POpossible)-set(HasPO))



def HallowedByP(PLACES,PasPL):
#
# Returns the possible rational places (for computing code vectors), provided that the PLACES (for constructing the linear system are given.
#
#
    R=list(set(PLACES)-set(PasPL))
    R.sort()
    return R

def GetRandomG(PLACES,NL2,GMAX=10,ALLOWED=[]):
#
#   Returns randomly chosen places (as a vector of length len(PLACES) from the allowed ones. 
#   There are NL2 entries in the vector, ant their values are randomely sampled from integers from one to GMAX.
#
#
#
    R=[0]*len(PLACES)
    #print(ALLOWED,PLACES)
    for i in random.sample(ALLOWED,min(NL2,len(ALLOWED))):
        #print(i)
        j=PLACES.index(i)
        R[j]=random.randint(1,GMAX)
    return R

def min_wt_rmat (M):
    m=len(M)
    n=len(M[0])
    Hwt=0
    for j in range(n):
        if M[0][j]!=0:
            Hwt=Hwt+1
    minHwt=Hwt
    sizerows=[0]*m
    sizerows[0]=Hwt
    k=0
    for i in range(1,m):
        Hwt=0
        for j in range(n):
            if M[i][j]!=0:
                Hwt=Hwt+1
        sizerows[i]=Hwt
        if Hwt<minHwt:
            minHwt=Hwt
            k=i
    
    return k,minHwt,sizerows

def MySubmat(intvec, M):
    A=[]
    for u in intvec:
        A.append(M[u])
    return A

def GetDeg(D,PLACES):
#
#  D is the divisor. Returns the degree \sum_i D[i]*deg(D[i])
#
    isum=0
    for i,u in enumerate(D):
        isum+=u*PLACES[i][0]
    return isum
    
def AdjustD(D,G0,size_D, size_D_min=1):
# 
# Adjusts D so that size_D_min<deg(D)<size_D
#
    n_nonzero=0
    for u in D:
        if u!=0:
            n_nonzero=n_nonzero+1
    while GetDeg(D,G0.PLACES)>=size_D or GetDeg(D,G0.PLACES)<=size_D_min:
        indexD=[i for i in range(len(D)) if D[i]>=1]
        j=random.sample(indexD,1)[0]
        if GetDeg(D,G0.PLACES)<=size_D_min:
            D[j]+=1
        elif GetDeg(D,G0.PLACES)>=size_D and D[j]>1:
            D[j]-=1
        if sum(D)==n_nonzero:
            break
        #print(D)
    return D
    
def DADJUST(D):
    NZ=[]
    for i,v in enumerate(D):
        if v!=0:
            NZ.append(i)
    if NZ!=[]:
        q=random.sample(NZ,1)
        D[q[0]]-=1
    return D
    
def Mutate(G0,lock=None):  
#
#  Returns the mutation of a gene G0. The divisor(s) are first redefined and the admissible places are then sampled again.
#
#  TODO: In the given gene, the lines to which the mutations are applied (intvec G, D, and m and others) should be chosen by some flags,
#  meanwhile, in the current version, the mutations are applied to all of them. (In other words, some of them should be locked.)
#
#  MODIFICATION:   
#  The dictionary lock determines whether the line of the gene that includes a is rewritten or not.
#
#        lock["ring s"] : Is the characteristic of the base ring rewritten? ~ currently not rewritable.
#        lock["Adj_div"]: Is the equation of the curve (the argment of Adj_div) rewritten? ~ currently not rewritable.
#        lock["intvec G"]: Is the divisor rewritten?
#        lock["intvec D"]: Are the places for computing code vectors rewritten?
#        lock["intvec m"]: Are the rows of the subcode in CSS construction rewritten? ~ In some cases, it might be neglected.
#    
#   Warning:
#   In some cases, the results provided by Singular returns the PLACES (not sorted), like
#   (2,1),(1,1),(1,2),(1,3),(2,2), (2,3)
#   To choose indeces of this array is To specify the divisors.
#
#   HOWEVER, TO SPECIFY THE RATIONAL PLACES FOR COMPUTING CODE VECTORS
#   THE DIVISORS AND THE RATIONAL PLACES MUST NOT HAVE COMMON ENTRIES.
#   TO THIS END, THE INDICES OF THE ARRAY
#   (1,1),(1,2),(1,3),(2,1,A),(2,1,B),(2,2,A),(2,2,B),(2,3,A),(2.3,B)
#   ARE CHOSEN. THE ARRAY IS SORTED AND CONTAINS THE MULTIPLE POINTS ACCORDING TO DEGREES.
#
    if lock==None:
        lock=dict()
        lock["ring s"]=False
        lock["Adj_div"]=False
        lock["intvec G"]=False
        lock["intvec D"]=False
        lock["intvec m"]=False
    AT=open("AT_.txt","w")
    iavoid=None
    HasPO=None
    HasPL=None
    PasPL=None
    AS=G0.gene_text.split("\n")
    for u in AS:
    # WARNINGS: G  should satisfy @math{ 2*genus-2 < deg(G) < size(D) }, which is
    #           not checked by the algorithm.
        size_D=config.size_D
        if u.find("ring s")>=0 and lock["ring s"]==False:
            #print("The base ring s is not mutable in the present version")
            print(u,file=AT)
        elif u.find("Adj_div")>=0 and lock["Adj_div"]==False:
            #print("The curve is not mutable in the present version")     
            print(u,file=AT)
        elif u.find("intvec G")>=0 and lock["intvec G"]==False:
            MODE_RANDOMLY_RESET=True
            #D=GetRandomG(G0.PLACES,2,GMAX=5,ALLOWED=G0.PLACES)
            #
            #  In some cases, G0.PLACES might include the entries with degree >=3,
            #  although the data for them are noy computed.
            #  To avoid to provide the divisor(s) with such PLACES, I remade the comment below.
            #
            D=GetRandomG(G0.PLACES,2,GMAX=5,ALLOWED=[u for u in G0.PLACES if u[0]<=2])
            if MODE_RANDOMLY_RESET==False:
                while GetDeg(D,G0.PLACES) >=size_D:
                    #D=GetRandomG(G0.PLACES,2,GMAX=5,ALLOWED=G0.PLACES)
                    D=GetRandomG(G0.PLACES,2,GMAX=5,ALLOWED=[u for u in G0.PLACES if u[0]<=2])
                    #print("D",D,GetDeg(D,G0.PLACES))
            else:
                D=AdjustD(D,G0,size_D)
            randomg=D
            J,HasPO,HasPL,PasPL=disj_divs (randomg,[l for l in range(1,G0.NrRatpl+1)],None,G0.NrRatpl,G0.PLACES)
            w=str(randomg).replace("]","").replace("[","")
            #print("intvec G="+w+";")     
            print("intvec G="+w+";",file=AT)
        elif u.find("intvec G")>=0 and lock["intvec G"]==True:
            ##
            ## This case may happen when we want to fix G and randomly choose D from aamissible ones. 
            ##
            randomg=eval("["+u.replace("intvec G=","").replace(";","")+"]")
            D=randomg
            J,HasPO,HasPL,PasPL=disj_divs (randomg,[l for l in range(1,G0.NrRatpl+1)],None,G0.NrRatpl,G0.PLACES)
            w=str(randomg).replace("]","").replace("[","")
            print("intvec G="+w+";",file=AT)          
        elif u.find("intvec D")>=0  and lock["intvec D"]==False:
            P2=copy.deepcopy(G0.PLACES)
            P2.sort()
            P2_TO_POINTS=dict()
            COUNTER=1
            for u in P2:
                u1,u2=u
                P2_TO_POINTS[u]=[COUNTER+w for w in range(u1)]
                COUNTER+=u1
            
            P2_TO_POINTS
            w=[]
            for u in set(G0.PLACES)-set(HasPL):
                if u[0]<=2:
                    w.extend(P2_TO_POINTS[u])
            w.sort()
            #print(w)
            #w=PallowedByH(HasPO,G0.NrRatpl)
            wr=random.sample(w,min(len(w),size_D))
            #print(wr,size_D)
            wr.sort()
            w=str(wr).replace("]","").replace("[","")
            DN=copy.deepcopy(D)
            DMODIFIED=False
            #if GetDeg(DN,G0.PLACES)>=len(wr):
            #    print("//WARN GetDeg(D,G0.PLACES)>=len(wr):",GetDeg(DN,G0.PLACES),DN,wr)
            #    print("//WARN GetDeg(D,G0.PLACES)>=len(wr):",GetDeg(DN,G0.PLACES),DN,wr,file=AT)
            while GetDeg(DN,G0.PLACES)>=len(wr):
            #    print("GetDeg(D,G0.PLACES)>=len(wr):",GetDeg(DN,G0.PLACES),DN,wr)
            #    #time.sleep(5)
                DN=DADJUST(DN)
            #    print("????GetDeg(D,G0.PLACES)>=len(wr):",GetDeg(DN,G0.PLACES),DN,wr) 
                DMODIFIED=True
            if DMODIFIED==True:
                DNS=str(DN).replace("]","").replace("[","")
                print("intvec G="+DNS+";",file=AT)       
            print("intvec D="+w+";",file=AT)

        #elif u.find("intvec m")>=0 and lock["intvec m"]==False:
        #    intvecm=random.sample([i for i in range(1,20)],2)
        #    w=str(intvecm).replace("]","").replace("[","")
        #    print("intvec m="+w+";",file=AT)
        else:
            print(u,file=AT)
    AT.close()

    AT=open("AT_.txt","r")
    gene_text=AT.read()
    AT.close()
    G=Gene(gene_text)
    # コマンド実行
    result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
    G.get_result(result.stdout)
    if result.stdout.find("Vector basis successfully computed")>=0:
        #print("Vector basis successfully computed")
        #print(result.stdout.split("\n")[-2])
        G.get_distance(int(result.stdout.split("\n")[-2]))
        OK=True
        return G
    else:
        OK=False
        G.get_distance(-9999)
        print("FAILED")
        #f=open("AT_.txt","r")
        #print(f.readline())
        #f.close()
        return G
    #print(result.stdout)     
    #print(result.stderr)   # エラー出力
    #print(result.returncode)  # 終了コード
    
def rewriteGene(TXT,KEY,VAL):
#
#  If the specified KEY is found in the gene text, the line including it is replaces by VAL.
#
    AS=TXT.split("\n")
    NTXT=""
    for u in AS:
    # WARNINGS: G  should satisfy @math{ 2*genus-2 < deg(G) < size(D) }, which is
    #           not checked by the algorithm.
        if u.find(KEY)>=0:
            #print("The base ring s is not mutable in the present version")
            NTXT+=VAL+"\n"
        else:
            NTXT+=u+"\n"
    if TXT==NTXT:
        print("WARNING: THE GENE IS NOT ALTERED AFTER REWRITING")
    return NTXT


def CROSSOVER(G1,G2):
    #print("CROSSOVER->")
    lock=dict()
    lock["ring s"]=True
    lock["Adj_div"]=False
    lock["intvec G"]=False
    lock["intvec D"]=False
    lock["intvec m"]=False
    g1_poly=G1.show_polynomial()
    g1_divisor=G1.show_divisor()
    g1_places=G1.show_places()
    #g1_CSS_rows=G1.show_CSS_rows()
    g2_poly=G2.show_polynomial()
    g2_divisor=G2.show_divisor()
    g2_places=G2.show_places()
    #g2_CSS_rows=G2.show_CSS_rows()  

    CHECK=[ g1_poly==g2_poly, g1_divisor==g2_divisor, g1_places==g2_places]
    print("CROSSOVER",CHECK)
    if CHECK[1]==False:
        #print(g1_divisor,g2_divisor)
        new_size=max(len(g1_divisor),len(g2_divisor))
        extended1 = np.zeros(new_size, dtype=np.array(g1_divisor).dtype)
        extended1[:len(g1_divisor)] = np.array(g1_divisor)
        extended2 = np.zeros(new_size, dtype=np.array(g2_divisor).dtype)
        extended2[:len(g2_divisor)] = np.array(g2_divisor)
        g12_divisor=[]
        for f1,f2 in zip (extended1,extended2):
            g12_divisor.append(random.sample([f1,f2],1)[0])
        intvecG12=[int(u) for u in g12_divisor]
        #print(intvecG12)
        New_GENE=rewriteGene(G1.gene_text,"intvec G","intvec G="+str(intvecG12).replace("]","").replace("[","")+";")
        GN=Gene(New_GENE)
        lock=dict()
        lock["ring s"]=True
        lock["Adj_div"]=True
        lock["intvec G"]=True
        lock["intvec D"]=False
        lock["intvec m"]=False
        #print("ADDITIONAL MUTATE")
        GN=Mutate(GN,lock)
        #print("<--CROSSOVER")
        return GN,lock
    if CHECK[2]==False:
        g12_places=g1_places+g2_places
        g12_places=list(set(g12_places))
        g12_places=random.sample(g12_places,min(len(g12_places),config.size_D))
        g12_places.sort()
        #print(g12_places)
        New_GENE=rewriteGene(G1.gene_text,"intvec D","intvec D="+str(g12_places).replace("]","").replace("[","")+";")
        GN=Gene(New_GENE)
        lock["ring s"]=True
        lock["Adj_div"]=True
        lock["intvec G"]=True
        lock["intvec D"]=True
        lock["intvec m"]=False
        #print("<---CROSSOVER")
        return GN,lock
    if CHECK[1]==True and CHECK[2]==True:
        return Mutate(G1,lock),lock
    return G1,lock

In [7]:
#
# Random Search: The loop goes on forever, and you should interrupt it anytime.
#
current_max_distance=0
OK=True
import time
G=Gene(A0+C)
print(vars(G))
OK=True
while OK==True:
    #RandomCSS_D_A()
    print(".",end="")
    
    G=Mutate(G)

    if G.distance>current_max_distance:
        current_max_distance=G.distance
        print(G.gene_text.replace("\n",""))
        print("distance=",G.distance,"current_max_distance=",current_max_distance)   # 標準出力


{'gene_text': 'LIB "mybrnoeth.lib";\nLIB "myprocs.txt";\nLIB "IsAinV.cpp";\nring s=2,(x,y),lp;\nlist HC=Adj_div(x3+y2+y);\nHC=NSplaces(1..2,HC);\nHC=extcurve(2,HC);\ndef ER=HC[1][4];\nsetring ER;\nintvec G=5;\nintvec D=2,3,4,5,6,7,8,9; \nmatrix C=AGcode_L(G,D,HC);\nprint(C);\nprint("Hamming_wt");\nprint(min_wt_rmat(C));\nprint("C");\nprint(C);\nprint("check_fully_one_rows");\nintvec a=check_fully_one_rows(C);\nprint(a);\nif (nrows(C)>1){C=MySubmat(a,C);\nprint("C [The row (1,1,...,1) is removed]");\nprint(C);\n}print("Check_fully_zero_columns");\nprint(check_fully_zero_columns(C));\nintvec b=check_fully_zero_columns(C);\nif (1){C=MySubmat_cols(b, C);\nprint("C [Fully zero colums are removed.]");\nprint(C);\n}print(my_min_wt_rmat(C));\nprint("SC");\nintvec m=1..nrows(C);\nm=shuffle(m,size(m));\nm=m[1],m[2];\nprint(m);\nmatrix SC=MySubmat(m,C);\nprint(SC);\nprint("Ker(C)");\nprint(MyKer(C));\nprint("Ker(SC)");\nprint(MyKer(SC));\nmatrix KC=MyKer(C);\nmatrix KSC=MyKer(SC);\nprint("KSC\\KC

KeyboardInterrupt: 

In [16]:
print(G.gene_text)

LIB "mybrnoeth.lib";
LIB "myprocs.txt";
LIB "IsAinV.cpp";
ring s=2,(x,y),lp;
list HC=Adj_div(x3+y2+y);
HC=NSplaces(1..2,HC);
HC=extcurve(2,HC);
def ER=HC[1][4];
setring ER;
intvec G=0, 0, 0, 0, 1, 2;
intvec G=0, 3, 0, 1, 0, 0;
intvec G=0, 0, 0, 1, 0, 2;
intvec G=4, 1, 0, 0, 0, 0;
intvec G=0, 0, 0, 1, 0, 2;
intvec G=1, 0, 0, 2, 0, 0;
intvec G=1, 0, 0, 1, 0, 0;
intvec G=0, 0, 1, 2, 0, 0;
intvec D=1, 2, 6, 7, 8, 9;
matrix C=AGcode_L(G,D,HC);
print(C);
print("Hamming_wt");
print(min_wt_rmat(C));
print("C");
print(C);
print("check_fully_one_rows");
intvec a=check_fully_one_rows(C);
print(a);
if (nrows(C)>1){C=MySubmat(a,C);
print("C [The row (1,1,...,1) is removed]");
print(C);
}print("Check_fully_zero_columns");
print(check_fully_zero_columns(C));
intvec b=check_fully_zero_columns(C);
if (1){C=MySubmat_cols(b, C);
print("C [Fully zero colums are removed.]");
print(C);
}print(my_min_wt_rmat(C));
print("SC");
intvec m=1..nrows(C);
m=Shuffle(m,size(m));
m=m[1],m[2];
print(m);
matrix SC=MySubm

In [17]:
f=open("./AT_ALL_SUBMAT.txt","r")
AT_ALL=f.read()
f.close()
AT_ALL=AT_ALL.replace("s=13","s=13")
#print(AT_ALL)

In [17]:
import random
from typing import List, Callable, Tuple

# 個体（遺伝子） = 整数のリスト
Individual = List[int]
Improvement = list()
Improvement =list()
class GeneticAlgorithm:
    def __init__(
        self,
        gene_length: int,
        gene_min: int,
        gene_max: int,
        population_size: int,
        fitness_func: Callable[[Individual], float],
        crossover_rate: float = 0.8,
        mutation_rate: float = 0.1,
        mutation_strength: int = 1,
    ):
        self.gene_length = gene_length
        self.gene_min = gene_min
        self.gene_max = gene_max
        self.population_size = population_size
        self.fitness_func = fitness_func
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.mutation_strength = mutation_strength

        self.population = self._initialize_population()

    # 初期集団生成
    def _initialize_population(self) -> List:
        G0=Gene(A0+C)
        return [
            Mutate(G0)
            for _ in range(self.population_size)
        ]

    # 評価
    def _evaluate(self) -> List[Tuple[Individual, float]]:
        return [(ind, self.fitness_func(ind)) for ind in self.population]

    # トーナメント選択
    def _selection(self) -> Individual:
        k = 3
        selected = random.sample(self.population, k)
        selected.sort(key=self.fitness_func, reverse=True)
        return selected[0]

    # 交叉（1点交叉）
    def _crossover(self, p1: Individual, p2: Individual) -> Individual:
        if random.random() > self.crossover_rate:
            return p1[:]

        point = random.randint(1, self.gene_length - 1)
        return p1[:point] + p2[point:]

    # 突然変異
    def _mutation(self, ind: Individual) -> Individual:
        new_ind = ind[:]
        for i in range(self.gene_length):
            if random.random() < self.mutation_rate:
                new_ind[i] += random.randint(
                    -self.mutation_strength, self.mutation_strength)
                # 範囲制限
                new_ind[i] = max(self.gene_min, min(self.gene_max, new_ind[i]))
        return new_ind

    # 1世代進化
    def step(self):
        new_population = []

        # エリート保存（最良個体1つ）
        evaluated = self._evaluate()
        best = max(evaluated, key=lambda x: x[1])[0]
        new_population.append(best)

        # 残り生成
        while len(new_population) < self.population_size:
            p1 = self._selection()
            p2 = self._selection()

            child = self._crossover(p1, p2)
            child = self._mutation(child)

            new_population.append(child)

        self.population = new_population

    def step_2A(self):
        # Only for testing, merely applying the mutation to genes.
        new_population = []
        global Improvement
        # エリート保存（最良個体1つ）-> TODO: Is it not better to keep all genes regarded as the best ones? 
        evaluated = self._evaluate()
        print(evaluated)
        best = max(evaluated, key=lambda x: x[1])[0]
        new_population.append(best)

        # 残り生成
        while len(new_population) < self.population_size:
            print("~",end="")
            p1 = self._selection()
            p2 = self._selection()
            p1gene={w:[u for u in p1.gene_text.split("\n") if u.find(w)>=0][0] for w in G.lock.keys()}
            p2gene={w:[u for u in p2.gene_text.split("\n") if u.find(w)>=0][0] for w in G.lock.keys()}
            print(p1gene,p2gene,difference_in_genes(p1gene,p2gene))
            child = random.sample([p1, p2],1)[0]
            child,lock = CROSSOVER(p1,p2)
            lock_all=dict()
            for ky in lock.keys():
                lock_all[ky]=True
            child =Mutate(child,lock=lock_all)
            current_distance=child.distance
            if child.distance > p1.distance and child.distance > p2.distance:
                Improvement.append(["CROSSOVER",child.distance,p1.distance,p2.distance])
            child = Mutate(child,lock=lock)
            if child.distance>current_distance:
                Improvement.append(["Mutate",child.distance,current_distance])                
            new_population.append(child)

        self.population = new_population    

    def step_2(self):
        # Only for testing, merely applying the mutation to genes.
        new_population = []
        global Improvement
        # エリート保存（最良個体1つ）-> TODO: Is it not better to keep all genes regarded as the best ones? 
        evaluated = self._evaluate()
        print(evaluated)
        best = max(evaluated, key=lambda x: x[1])[0]
        new_population.append(best)

        # 残り生成
        while len(new_population) < self.population_size:
            #print("~",end="")
            p1 = self._selection()
            p2 = self._selection()
            p1gene={w:[u for u in p1.gene_text.split("\n") if u.find(w)>=0][0] for w in G.lock.keys()}
            p2gene={w:[u for u in p2.gene_text.split("\n") if u.find(w)>=0][0] for w in G.lock.keys()}
            #print(p1gene,p2gene,difference_in_genes(p1gene,p2gene))
            lock=dict()
            lock["ring s"]=False
            lock["Adj_div"]=False
            lock["intvec G"]=False
            lock["intvec D"]=False
            lock["intvec m"]=False
            child=None
            if random.random()> config.crossover_rate:
                #print("NO CROSSOVER")
                child = random.sample([p1, p2],1)[0]
            else:
                print("CROSSOVER")
                child,lock = CROSSOVER(p1,p2)
            #lock_all=dict()
            #for ky in lock.keys():
            #    lock_all[ky]=True
            #child =Mutate(child,lock=lock_all)
            #new_population.append(child)
            #current_distance=child.distance
            #if child.distance > p1.distance and child.distance > p2.distance:
            #    Improvement.append(["CROSSOVER",child.distance,p1.distance,p2.distance])
            if random.random()<config.mutation_rate:
                print("Mutate After CROSSING",lock)
                child = Mutate(child,lock=lock)
                #if child.distance>current_distance:
                #    Improvement.append(["Mutate",child.distance,current_distance])     
            if child!=None:
                new_population.append(child)
            print(len(new_population))

        self.population = new_population    

    
    # 実行
    def run(self, generations: int, verbose: bool = True):
        for gen in range(generations):
            print(".",end="")
            self.step_2()
            best, fitness = self.get_best()

            if verbose:
                print(f"Gen {gen}: Best Fitness = {fitness}")

        return self.get_best()

    # 最良個体取得
    def get_best(self) -> Tuple[Individual, float]:
        evaluated = self._evaluate()
        evaluated=[u for u in evaluated if u!=None]             
        return max(evaluated, key=lambda x: x[1])

def fitness(individual):
    return individual.distance

def difference_in_genes(p1,p2):
    differs=list()
    for ky in p1.keys():
        if p1[ky]!=p2[ky]:
            differs.append(ky)
    return differs
    


In [18]:
ga = GeneticAlgorithm(
    gene_length=10,
    gene_min=0,
    gene_max=10,
    population_size=10,
    fitness_func=fitness
)
[(ind, fitness(ind)) for ind in ga.population]
ga.population=[u for u in ga.population if u.distance<2]

In [19]:
[(ind, fitness(ind)) for ind in ga.population]

[(<__main__.Gene at 0x7a3a44ff2450>, 1),
 (<__main__.Gene at 0x7a3a44ff2510>, 1),
 (<__main__.Gene at 0x7a3a44ff3410>, 1),
 (<__main__.Gene at 0x7a3a44ff0560>, 1),
 (<__main__.Gene at 0x7a3a44ff31a0>, 1),
 (<__main__.Gene at 0x7a3a44ff24b0>, 1)]

In [20]:
print(ga.population[-1].result)

// ** redefining closed_points (LIB "myprocs.txt";) AT_.txt:2
// ** redefining Adj_div (LIB "myprocs.txt";) AT_.txt:2
// ** redefining NSplaces (LIB "myprocs.txt";) AT_.txt:2
// ** redefining BrillNoether (LIB "myprocs.txt";) AT_.txt:2
// ** redefining Weierstrass (LIB "myprocs.txt";) AT_.txt:2
// ** redefining permute_L (LIB "myprocs.txt";) AT_.txt:2
// ** redefining dual_code (LIB "myprocs.txt";) AT_.txt:2
// ** redefining AGcode_L (LIB "myprocs.txt";) AT_.txt:2
// ** redefining AGcode_Omega (LIB "myprocs.txt";) AT_.txt:2
// ** redefining extcurve (LIB "myprocs.txt";) AT_.txt:2
// ** redefining prepSV (LIB "myprocs.txt";) AT_.txt:2
// ** redefining decodeSV (LIB "myprocs.txt";) AT_.txt:2
// ** redefining sys_code (LIB "myprocs.txt";) AT_.txt:2
Computing affine singular points ... 
Computing all points at infinity ... 
Computing affine singular places ... 
Computing singular places at infinity ... 
Computing non-singular places at infinity ... 
Adjunction divisor computed successfully

In [ ]:
best_individual, best_score = ga.run(generations=100)

print("Best:", best_individual)
print("Score:", best_score)

.[(<__main__.Gene object at 0x7a3a44ff2450>, 1), (<__main__.Gene object at 0x7a3a44ff2510>, 1), (<__main__.Gene object at 0x7a3a44ff3410>, 1), (<__main__.Gene object at 0x7a3a44ff0560>, 1), (<__main__.Gene object at 0x7a3a44ff31a0>, 1), (<__main__.Gene object at 0x7a3a44ff24b0>, 1)]
CROSSOVER
CROSSOVER [True, False, False]
Mutate After CROSSING {'ring s': True, 'Adj_div': True, 'intvec G': True, 'intvec D': False, 'intvec m': False}
2
Mutate After CROSSING {'ring s': False, 'Adj_div': False, 'intvec G': False, 'intvec D': False, 'intvec m': False}
3
CROSSOVER
CROSSOVER [True, False, True]
4
CROSSOVER
CROSSOVER [True, False, True]
Mutate After CROSSING {'ring s': True, 'Adj_div': True, 'intvec G': True, 'intvec D': False, 'intvec m': False}
5
CROSSOVER
CROSSOVER [True, False, False]
Mutate After CROSSING {'ring s': True, 'Adj_div': True, 'intvec G': True, 'intvec D': False, 'intvec m': False}
6
CROSSOVER
CROSSOVER [True, False, False]
Mutate After CROSSING {'ring s': True, 'Adj_div': Tr

In [13]:
ga.population[0].show_places()

[1, 2, 6, 7, 8, 9]

In [13]:
ga = GeneticAlgorithm(
    gene_length=10,
    gene_min=0,
    gene_max=10,
    population_size=10,
    fitness_func=fitness
)
[(ind, fitness(ind)) for ind in ga.population]

[(<__main__.Gene at 0x708efc59e780>, 1),
 (<__main__.Gene at 0x708efc4a46e0>, 1),
 (<__main__.Gene at 0x708ee170b500>, 1),
 (<__main__.Gene at 0x708ee170bef0>, 1),
 (<__main__.Gene at 0x708ee170ba70>, 1),
 (<__main__.Gene at 0x708ee170bb30>, 1),
 (<__main__.Gene at 0x708ee1709490>, 1),
 (<__main__.Gene at 0x708ee170b980>, 1),
 (<__main__.Gene at 0x708ee170b830>, 1),
 (<__main__.Gene at 0x708ee170bb90>, 1)]

In [19]:
ga.population[0].show_polynomial()

'x3+y2+y'

In [18]:
#
# A quality check of the genes
#
for u in ga.population:
    print(0<GetDeg(u.show_divisor(),u.PLACES)<len(u.show_places()))

True
True
True
True
True
True
True
True
True
True


In [14]:
Improvement

[]

In [15]:
best_individual, best_score = ga.run(generations=100)

print("Best:", best_individual)
print("Score:", best_score)

.[(<__main__.Gene object at 0x708efc59e780>, 1), (<__main__.Gene object at 0x708efc4a46e0>, 1), (<__main__.Gene object at 0x708ee170b500>, 1), (<__main__.Gene object at 0x708ee170bef0>, 1), (<__main__.Gene object at 0x708ee170ba70>, 1), (<__main__.Gene object at 0x708ee170bb30>, 1), (<__main__.Gene object at 0x708ee1709490>, 1), (<__main__.Gene object at 0x708ee170b980>, 1), (<__main__.Gene object at 0x708ee170b830>, 1), (<__main__.Gene object at 0x708ee170bb90>, 1)]
Mutate After CROSSING {'ring s': False, 'Adj_div': False, 'intvec G': False, 'intvec D': False, 'intvec m': False}
2
CROSSOVER
CROSSOVER [True, False, False]
Mutate After CROSSING {'ring s': True, 'Adj_div': True, 'intvec G': True, 'intvec D': False, 'intvec m': False}
3
CROSSOVER
CROSSOVER [True, True, True]
Mutate After CROSSING {'ring s': True, 'Adj_div': False, 'intvec G': False, 'intvec D': False, 'intvec m': False}
4
CROSSOVER
CROSSOVER [True, False, False]
Mutate After CROSSING {'ring s': True, 'Adj_div': True, 'int

KeyboardInterrupt: 

In [55]:
ga.population[3].show_gene()

LIB "mybrnoeth.lib";
LIB "myprocs.txt";
ring s=2,(x,y),lp;
list HC=Adj_div(x3+y2+y);
HC=NSplaces(1..2,HC);
HC=extcurve(2,HC);
def ER=HC[1][4];
setring ER;
intvec G=1, 0, 0, 1, 0, 0;
intvec G=1, 0, 0, 1, 0, 0;
intvec G=1, 0, 0, 1, 0, 0;
intvec D=2, 3, 6, 7, 8, 9;
matrix C=AGcode_L(G,D,HC);
print(C);
print("Hamming_wt");
print(min_wt_rmat(C));
print("C");
print(C);
print("check_fully_one_rows");
intvec a=check_fully_one_rows(C);
print(a);
if (nrows(C)>1){C=MySubmat(a,C);
print("C [The row (1,1,...,1) is removed]");
print(C);
}print("Check_fully_zero_columns");
print(check_fully_zero_columns(C));
intvec b=check_fully_zero_columns(C);
if (1){C=MySubmat_cols(b, C);
print("C [Fully zero colums are removed.]");
print(C);
}print(my_min_wt_rmat(C));
print("SC");
intvec m=3, 8;
matrix SC=MySubmat(m,C);
print(SC);
print("Ker(C)");
print(MyKer(C));
print("Ker(SC)");
print(MyKer(SC));
matrix KC=MyKer(C);
matrix KSC=MyKer(SC);
print("KSC\KC");
print(MySupplement(KC,KSC));
print(my_min_wt_rmat(MySupp